# 📊 Visualización de Datos y Evaluación del Modelo

En este notebook vamos a explorar visualmente el dataset que generamos mediante nuestro Feature Engineering y a interpretar el modelo predictivo de Prophetia2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Configuración de estilo estético para los gráficos
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", palette="pastel")

# Cargar datos
df = pd.read_parquet('../data/processed/matches_dataset.parquet')
print(f"Dataset cargado con dimensiones: {df.shape}")

## 1. Análisis de Goles Esperados (xG) vs Resultado
Veamos si los equipos que ganaron realmente generaron más xG y concedieron menos.

In [ ]:
plt.figure(figsize=(10, 6))

# Mapear outcome para la leyenda
outcome_map = {1: 'Victoria', 0: 'Empate', -1: 'Derrota'}
df['Resultado_Str'] = df['outcome'].map(outcome_map)

sns.scatterplot(data=df, x='xg_created', y='xg_conceded', hue='Resultado_Str', 
                palette={'Victoria': '#2ecc71', 'Empate': '#f1c40f', 'Derrota': '#e74c3c'}, 
                s=100, alpha=0.8)

plt.title('Relación entre xG Creado vs xG Concedido según el Resultado del Partido', fontsize=14)
plt.xlabel('Goles Esperados (xG) a Favor')
plt.ylabel('Goles Esperados (xG) en Contra')
plt.axline((0, 0), slope=1, color="white", linestyle="--", alpha=0.5, label="xG a Favor = xG en Contra")
plt.legend(title="Resultado")
plt.show()

## 2. Correlación de Variables Tácticas
¿Qué variables tienen una mayor correlación matemática con la victoria (`outcome`)?

In [ ]:
cols_of_interest = ['outcome', 'xg_created', 'xg_conceded', 'shots_on_target', 
                    'possession_pct', 'corners', 'passes_completed', 'pressures']

corr = df[cols_of_interest].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Matriz de Correlación de Métricas Tácticas', fontsize=15)
plt.show()

## 3. ¿Qué vio el Modelo? (Feature Importance)
Cargamos el modelo entrenado para graficar qué variables considera más importantes para tomar decisiones.

In [ ]:
try:
    saved_data = joblib.load('../core/save_models/prophetia_xgb_model.pkl')
    model = saved_data['model']
    feature_cols = saved_data['features']
    
    # Extraer el XGBoost y su selector del VotingClassifier/Pipeline
    calibrated_classifiers = model.calibrated_classifiers_
    voting_clf = calibrated_classifiers[0].estimator
    pipeline_xgb = voting_clf.estimators_[0]
    xgb_model = pipeline_xgb.named_steps['clf']
    selector = pipeline_xgb.named_steps['feature_selection']
    
    selected_mask = selector.get_support()
    selected_features = np.array(feature_cols)[selected_mask]
    
    importances = xgb_model.feature_importances_
    indices = np.argsort(importances)[-10:] # Top 10
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), importances[indices], color='#3498db', align='center')
    plt.yticks(range(len(indices)), [selected_features[i] for i in indices])
    plt.xlabel('Importancia Relativa')
    plt.title('Top 10 Variables más importantes para el Modelo (XGBoost)', fontsize=14)
    plt.show()
except Exception as e:
    print(f"Error al visualizar modelo: {e}")